# Set up

## Change working directory

In [1]:
%cd ../

C:\Users\Skylar\Downloads\tools


## Install packages

In [2]:
!pip install -q sympy

## Import development libraries

In [3]:
import io
import string
import tempfile
import textwrap
import time

import hypothesis
import ipytest
import pytest
import sympy
from hypothesis import strategies as st
from hypothesis.extra import numpy as hseanp
from PIL import Image

ipytest.autoconfig(rewrite_asserts=False)

## Import other libraries

In [4]:
import contextlib
import datetime
import logging
import pathlib
import shutil
import typing
import warnings

import numpy as np
import pandas as pd
import toolz as tz
import yaml
from matplotlib import pyplot as plt

## Declare constants

In [5]:
module: str = "utils"

# Define DocstringParser

## Define

In [6]:
class DocstringParser:
    def __init__(self, x: typing.Any) -> None:
        self.docstring: str = x.__doc__ or ""

    def fit(self) -> "DocstringParser":
        self.docstring_lines: pd.Series = self._get_lines()
        self.docstring_sections: pd.DataFrame = self._get_sections()
        return self

    def print_section(self, section: str = "Overview") -> None:
        print(self._get_section(section=section), end="\n" * 2)

    def print_sections(self, sections: list[str] | tuple[str] = ("Overview",)) -> None:
        for section in sections:
            self.print_section(section=section)

    def _prefix(self) -> str:
        return "Overview\n--------\n%s" % self.docstring

    def _get_lines(self) -> pd.Series:
        return pd.Series(data=self._prefix().splitlines(), name="lines").str.strip()

    def _get_sections(self) -> pd.DataFrame:
        assign_args: dict[str, typing.Callable] = {
            "has_underscores": lambda x: x["lines"].str.contains(pat=r"^[-=]{3,}$"),
            "is_header": lambda x: x["has_underscores"].shift(periods=-1).ffill(),
            "headers": lambda x: x["lines"].where(cond=x["is_header"]).ffill().fillna(value="Overview"),
        }
        agg_args: dict[str, typing.Callable] = {
            "starts": lambda x: x.index.min(),
            "stops": lambda x: x.index.max().__add__(1),
        }
        return (
            self.docstring_lines.to_frame()
            .assign(**assign_args)
            .groupby(by="headers")["lines"]
            .agg(**agg_args)
            .sort_values(by="starts")
        )

    def _get_section(self, section: str) -> str:
        if section not in self.docstring_sections.index:
            raise ValueError("Unknown section: %s" % section)
        starts_and_stops: pd.Series = self.docstring_sections.loc[section, :]
        return self.docstring_lines.iloc[slice(*starts_and_stops)].str.cat(sep="\n")

## Demonstrate

In [7]:
parser = DocstringParser(x=pd.DataFrame).fit()
display(parser.docstring_sections)
parser.print_sections(sections=["Overview", "Examples"])

C:\Users\Skylar\AppData\Local\Temp\ipykernel_16416\4269669346.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "is_header": lambda x: x["has_underscores"].shift(periods=-1).ffill(),


,starts,stops
headers,,
Overview,0,10
Parameters,10,39
See Also,39,47
Notes,47,51
Examples,51,135


Overview
--------

Two-dimensional, size-mutable, potentially heterogeneous tabular data.

Data structure also contains labeled axes (rows and columns).
Arithmetic operations align on both row and column labels. Can be
thought of as a dict-like container for Series objects. The primary
pandas data structure.


Examples
--------
Constructing DataFrame from a dictionary.

>>> d = {'col1': [1, 2], 'col2': [3, 4]}
>>> df = pd.DataFrame(data=d)
>>> df
col1  col2
0     1     3
1     2     4

Notice that the inferred dtype is int64.

>>> df.dtypes
col1    int64
col2    int64
dtype: object

To enforce a single dtype:

>>> df = pd.DataFrame(data=d, dtype=np.int8)
>>> df.dtypes
col1    int8
col2    int8
dtype: object

Constructing DataFrame from a dictionary including Series:

>>> d = {'col1': [0, 1, 2, 3], 'col2': pd.Series([2, 3], index=[2, 3])}
>>> pd.DataFrame(data=d, index=[0, 1, 2, 3])
col1  col2
0     0   NaN
1     1   NaN
2     2   2.0
3     3   3.0

Constructing DataFrame from numpy nda

## Test

In [8]:
%%ipytest

# Helper(s)
def make_parser(doc: str) -> DocstringParser:
    class X:
        __doc__ = doc

    return DocstringParser(x=X).fit()

# Deterministic test(s)
def test_has_specified_sections() -> None:
    doc: str = """
    Summary
    -------
    Something here.
    
    Parameters
    ----------
    x : int
        description
    """
    parser: DocstringParser = make_parser(doc=textwrap.dedent(text=doc))
    assert set(parser.docstring_sections.index) == {"Overview", "Summary", "Parameters"}


def test_does_unknown_section_raise() -> None:
    parser: DocstringParser = make_parser(doc="Summary")
    with pytest.raises(expected_exception=ValueError):
        parser._get_section(section="Not a Section")

..                                                                                           [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

t_f037dbe23fda42e0bdccb41e807ae466.py::test_has_specified_sections
t_f037dbe23fda42e0bdccb41e807ae466.py::test_does_unknown_section_raise
  C:\Users\Skylar\AppData\Local\Temp\ipykernel_16416\4269669346.py:26: Fu

# Define filter_dir

## Define

In [9]:
def filter_dir(x: typing.Any, include_underscores: bool = False, include_modules: bool = False) -> pd.DataFrame:
    assign_args: dict[str, typing.Callable] = {
        "has_underscore": lambda y: y["object"].str.startswith(pat="_"),
        "type": lambda y: y["object"].apply(func=lambda z: type(getattr(x, z)).__name__),
        "is_module": lambda y: y["type"].eq(other="module"),
    }
    return (
        pd.Series(data=dir(x))
        .to_frame(name="object")
        .assign(**assign_args)
        .query(expr="has_underscore.eq(other=%s)" % include_underscores)
        .query(expr="is_module.eq(other=%s)" % include_modules)
        .set_index(keys="object")
    )

## Demonstrate

In [10]:
dir_data: list[pd.DataFrame] = []
for include_underscores in [False, True]:
    for include_modules in [False, True]:
        args = dict(include_underscores=include_underscores, include_modules=include_modules)
        dir_data.append(filter_dir(x=pd, **args))
display(*dir_data)

,has_underscore,type,is_module
object,,,
ArrowDtype,False,type,False
BooleanDtype,False,type,False
Categorical,False,ABCMeta,False
CategoricalDtype,False,type,False
CategoricalIndex,False,type,False
...,...,...,...
to_pickle,False,function,False
to_timedelta,False,function,False
unique,False,function,False


,has_underscore,type,is_module
object,,,
api,False,module,True
arrays,False,module,True
compat,False,module,True
core,False,module,True
errors,False,module,True
io,False,module,True
offsets,False,module,True
pandas,False,module,True
plotting,False,module,True


,has_underscore,type,is_module
object,,,
__all__,True,list,False
__builtins__,True,dict,False
__cached__,True,str,False
__doc__,True,str,False
__docformat__,True,str,False
__file__,True,str,False
__git_version__,True,str,False
__loader__,True,SourceFileLoader,False
__name__,True,str,False


,has_underscore,type,is_module
object,,,
_config,True,module,True
_libs,True,module,True
_testing,True,module,True
_typing,True,module,True
_version_meson,True,module,True


## Test

In [11]:
%%ipytest

base: typing.Any = st.one_of(st.floats(), st.integers(), st.none(), st.text())
containers: typing.Any = st.recursive(
    base=base,
    extend=lambda x: st.one_of(st.dictionaries(keys=st.text(), values=x), st.lists(elements=x), st.tuples(x, x)),
)
random_object: typing.Any = st.one_of(base, containers)

@hypothesis.given(
    x=random_object,
    include_underscores=st.booleans(),
    include_modules=st.booleans()
)
def test_filter_dir(x: typing.Any, include_underscores: bool, include_modules: bool) -> None:
    out: pd.DataFrame = filter_dir(x=x, include_underscores=include_underscores, include_modules=include_modules)
    if not out.empty:
        assert out["has_underscore"].any() == include_underscores
        assert out["is_module"].any() == include_modules

.                                                                                            [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 2.28s


# Define get_shape

## Define

In [12]:
def get_shape(x: typing.Any) -> typing.Any:
    if hasattr(x, "shape"):
        return x.shape
    elif hasattr(x, "size"):
        return x.size
    elif hasattr(x, "__len__"):
        return len(x)
    else:
        return np.nan

## Demontrate

In [13]:
get_shape(x=np.random.randn(8, 4))

(8, 4)

In [14]:
get_shape(x=Image.new(mode="RGB", size=(8, 4)))

(8, 4)

In [15]:
get_shape(x=np.random.randn(8, 4).tolist())

8

In [16]:
get_shape(x=8)

nan

## Test

In [17]:
%%ipytest

arrays: np.ndarray = hseanp.arrays(
    dtype=float, shape=st.tuples(st.integers(min_value=0, max_value=9), st.integers(min_value=0, max_value=9))
)
random_object: typing.Any = st.one_of(base, containers, arrays)

@hypothesis.given(random_object)
def test_get_shape(x: typing.Any) -> None:
    out: typing.Any = get_shape(x=x)
    if hasattr(x, "shape"):
        assert out == x.shape
    elif hasattr(x, "size"):
        assert out == x.size
    elif hasattr(x, "__len__"):
        assert out == len(x)
    else:
        assert np.isnan(out)

.                                                                                            [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 0.91s


# Define get_type_and_shape

## Define

In [18]:
def get_type_and_shape(x: typing.Any) -> tuple[type, typing.Any]:
    return type(x), get_shape(x=x)

## Demonstrate

In [19]:
get_type_and_shape(x=np.random.randn(8, 4))

(numpy.ndarray, (8, 4))

In [20]:
get_type_and_shape(x=Image.new(mode="RGB", size=(8, 4)))

(PIL.Image.Image, (8, 4))

In [21]:
get_type_and_shape(x=np.random.randn(8, 4).tolist())

(list, 8)

In [22]:
get_type_and_shape(x=8)

(int, nan)

## Test

In [23]:
%%ipytest

@hypothesis.given(random_object)
def test_get_shape(x: typing.Any) -> None:
    out: tuple[type, typing.Any] = get_type_and_shape(x=x)
    assert out[0] is type(x)
    if hasattr(x, "shape"):
        assert out[1] == x.shape
    elif hasattr(x, "size"):
        assert out[1] == x.size
    elif hasattr(x, "__len__"):
        assert out[1] == len(x)
    else:
        assert np.isnan(out[1])

.                                                                                            [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 0.58s


# Define describe_structure

## Define

In [24]:
def describe_structure(x: typing.Any, indent: int = 0, max_indent: int = 2) -> None:
    prefix: str = "  " * indent
    if indent > max_indent:
        print("%s(Depth limit reached)" % prefix)
        return
    if isinstance(x, dict):
        print("%s%s with %d keys" % (prefix, type(x), len(x)))
        for key, value in x.items():  # type: tuple[typing.Any, typing.Any]
            print("%s- key: %s" % (prefix, key))
            describe_structure(x=value, indent=indent + 1, max_indent=max_indent)
    elif isinstance(x, (list, set, tuple)):
        print("%s%s with %d elements" % (prefix, type(x), len(x)))
        for i, element in enumerate(iterable=x):  # type: tuple[int, typing.Any]
            print("%s- element: %d" % (prefix, i))
            describe_structure(x=element, indent=indent + 1, max_indent=max_indent)
    else:
        print("%s%s" % (prefix, get_type_and_shape(x=x)))

## Demonstrate

In [25]:
nested_structure: dict[str, typing.Any] = {
    "info": {
        "name": "demo",
        "tags": ["a", "b"],
    },
    "values": [1, 2, [3, 4]],
}
describe_structure(x=nested_structure)

<class 'dict'> with 2 keys
- key: info
  <class 'dict'> with 2 keys
  - key: name
    (<class 'str'>, 4)
  - key: tags
    <class 'list'> with 2 elements
    - element: 0
      (Depth limit reached)
    - element: 1
      (Depth limit reached)
- key: values
  <class 'list'> with 3 elements
  - element: 0
    (<class 'int'>, nan)
  - element: 1
    (<class 'int'>, nan)
  - element: 2
    <class 'list'> with 2 elements
    - element: 0
      (Depth limit reached)
    - element: 1
      (Depth limit reached)


## Test

In [26]:
%%ipytest

def capture(fn: typing.Callable) -> str:
    buffer = io.StringIO()
    with contextlib.redirect_stdout(new_target=buffer):
        fn()
    return buffer.getvalue()
    

@hypothesis.given(random_object)
def test_describe_structure(x: typing.Any):
    out = capture(fn=lambda: describe_structure(x=x, max_indent=1))
    assert "<class" in out or "(Depth limit reached)" in out

.                                                                                            [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 2 warnings in 0.91s


# Define display_data

## Define

In [27]:
def display_data(
    data: pd.DataFrame | pd.Series, is_in_notebook: bool, name: str | None = None, info_args: dict | None = None
) -> None:
    if name is not None:
        print("=" * int(8e1), name, "-" * int(8e1), sep="\n")
    data.info(**(info_args or {}))
    if is_in_notebook:
        print("-" * int(8e1))
        display(data)

## Demonstrate

In [28]:
data = pd.DataFrame(data={"a": list("xyz"), "b": range(3)})
display_data(data=data, is_in_notebook=True, name="Example data", info_args=dict(verbose=True))

Example data
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   a       3 non-null      object
 1   b       3 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 180.0+ bytes
--------------------------------------------------------------------------------


,a,b
0,x,0
1,y,1
2,z,2


## Test

In [29]:
%%ipytest

@pytest.mark.parametrize('name', ['Example data', None])
def test_name(name: str | None, capsys) -> None:
    data = pd.DataFrame(data={'a': list('xyz')})
    display_data(data=data, is_in_notebook=False, name=name)
    out = capsys.readouterr().out
    if name:
        assert name in out
    else:
        assert '=' not in out

..                                                                                           [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
2 passed, 2 warnings in 0.03s


# Define print_shapes

## Define

In [30]:
def print_shapes(x: typing.Any, include_types: bool = False, **kwargs) -> None:
    print(*map(get_type_and_shape if include_types else get_shape, x), **kwargs)

## Demonstrate

In [31]:
x: list[typing.Any] = [1, [2, 3, 4], np.random.randn(5, 6)]
for include_types in [False, True]:
    print_shapes(x=x, include_types=include_types, sep="\n", end="\n" + "-" * int(8e1) + "\n")

nan
3
(5, 6)
--------------------------------------------------------------------------------
(<class 'int'>, nan)
(<class 'list'>, 3)
(<class 'numpy.ndarray'>, (5, 6))
--------------------------------------------------------------------------------


## Test

In [32]:
# Placeholder

# Define ignore_exceptions

## Define

In [33]:
@contextlib.contextmanager
def ignore_exceptions() -> typing.Iterator[None]:
    try:
        yield
    except Exception as exception:
        print(type(exception), exception)

## Demonstrate

In [34]:
with ignore_exceptions():
    1 / 0

<class 'ZeroDivisionError'> division by zero


## Test

In [35]:
%%ipytest

# Deterministic test(s)
def test_is_error_ignored(capsys) -> None:
    with ignore_exceptions():
        raise ValueError('This is value error')

    capture_result = capsys.readouterr().out
    assert 'ValueError' in capture_result
    assert 'This is value error' in capture_result

def test_is_non_error_allowed(capsys) -> None:
    with ignore_exceptions():
        1 + 1

    capture_result = capsys.readouterr().out
    assert capture_result == ""

..                                                                                           [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
2 passed, 2 warnings in 0.02s


# Define ignore_warnings

## Define

In [36]:
@contextlib.contextmanager
def ignore_warnings() -> typing.Iterator[None]:
    with warnings.catch_warnings():
        warnings.filterwarnings(action="ignore")
        yield

## Demonstrate

In [37]:
warnings.warn(message="This is deprecated", category=DeprecationWarning)

C:\Users\Skylar\AppData\Local\Temp\ipykernel_16416\1510999668.py:1: DeprecationWarning: This is deprecated
  warnings.warn(message="This is deprecated", category=DeprecationWarning)


In [38]:
with ignore_warnings():
    warnings.warn(message="This is deprecated", category=DeprecationWarning)

## Test

In [39]:
%%ipytest

# Deterministic test(s)
def test_is_non_warning_allowed(capsys) -> None:
    with ignore_warnings():
        x = 1 + 1

    assert x == 2

# Property-based test(s)
@hypothesis.given(
    message=st.text(),
    category=st.sampled_from(elements=[DeprecationWarning, RuntimeWarning, UserWarning])
)
def test_is_warning_ignored(message: str, category: typing.Any) -> None:
    buffer = io.StringIO()
    with contextlib.redirect_stdout(new_target=buffer):
        with ignore_warnings():
            warnings.warn(message=message, category=category)

    assert buffer.getvalue() == ''

..                                                                                           [100%]
======================================== warnings summary =========================================
env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; _hypothesis_globals
    self._mark_plugins_for_rewrite(hook, disable_autoload)

env\Lib\site-packages\_pytest\config\__init__.py:1309
  C:\Users\Skylar\Downloads\tools\env\Lib\site-packages\_pytest\config\__init__.py:1309: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; hypothesis
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
2 passed, 2 warnings in 0.12s


# Define print_sequence

## Define

In [40]:
def print_sequence(x: typing.Any, header: str = "Sequence") -> None:
    length: int = len(x)
    length_of_length: int = len(str(length))
    sequence: str = tz.pipe(
        enumerate(iterable=x), tz.curried.map(lambda y: f"{y[0]:0{length_of_length}d}. {y[1]}"), "\n".join
    )
    print("%s (%d):\n%s" % (header, length, sequence))

## Demonstrate

In [41]:
print_sequence(x=string.ascii_lowercase, header="Lowercase letters")

Lowercase letters (26):
00. a
01. b
02. c
03. d
04. e
05. f
06. g
07. h
08. i
09. j
10. k
11. l
12. m
13. n
14. o
15. p
16. q
17. r
18. s
19. t
20. u
21. v
22. w
23. x
24. y
25. z


## Test

In [42]:
# Placeholder

# Define print_type_and_return

## Define

In [43]:
def print_type_and_return(x: typing.Any) -> typing.Any:
    print(type(x))
    return x

## Demonstrate

In [44]:
x, t = sympy.symbols(names="x t")  # type: tuple[sympy.Symbol, sympy.Symbol]
A = sympy.Matrix([[x, 1], [0, x**2]])
expr: typing.Any = sympy.exp(A) * sympy.integrate(sympy.sin(x) / x, (x, 0, t))
print_type_and_return(x=expr)

<class 'sympy.matrices.immutable.ImmutableDenseMatrix'>


Matrix([
[exp(x)*Si(t), (-exp(x)/(x**2 - x) - exp(x**2)/(-x**2 + x))*Si(t)],
[           0,                                    exp(x**2)*Si(t)]])

## Test

In [45]:
# Placeholder

# Define read_in_params

## Define

In [46]:
def read_in_params(path: str) -> dict[str, typing.Any]:
    with open(file=path) as file:
        params: dict[str, typing.Any] = yaml.safe_load(stream=file)
    return params

## Demonstrate

In [47]:
params_path = pathlib.Path("params.yaml")

In [48]:
%%file $params_path

basic_config_args:
    format: '%(asctime)s - %(levelname)s - %(msg)s'
    level: 20

Writing params.yaml


In [49]:
params: dict[str, typing.Any] = read_in_params(path=params_path)
params_path.unlink()
params

{'basic_config_args': {'format': '%(asctime)s - %(levelname)s - %(msg)s',
  'level': 20}}

## Test

In [50]:
# Placeholder

# Define configure_logging

## Define

In [51]:
def configure_logging(path: str | None, params: dict[str, typing.Any]) -> pathlib.Path | None:
    if path is not None:
        outputs_directory = pathlib.Path(path)
        shutil.rmtree(path=outputs_directory, ignore_errors=True)
        outputs_directory.mkdir()
        logging.basicConfig(filename=outputs_directory / str(datetime.date.today()), **params["basic_config_args"])
        return outputs_directory
    else:
        logging.basicConfig(**params["basic_config_args"])

## Demonstrate

In [52]:
configure_logging(path=None, params=params)
logging.info(msg="Demonstrate")

2026-05-19 17:08:20,043 - INFO - Demonstrate


## Test

In [53]:
# Placeholder

# Define save_show_and_close

## Define

In [54]:
def save_show_and_close(outputs_directory: pathlib.Path | None, filename: str | None, is_in_notebook: bool) -> None:
    if outputs_directory is not None and filename is not None:
        plt.savefig(fname=outputs_directory / filename, bbox_inches="tight")
    if is_in_notebook:
        plt.show()
    plt.close()

## Demonstrate

In [55]:
# Placeholder

## Test

In [56]:
# Placeholder

# Define time_callable

## Define

In [57]:
def time_callable(fn: typing.Callable) -> typing.Callable:
    def wrap_callable(*args, **kwargs):
        now = datetime.datetime.now()
        result = fn(*args, **kwargs)
        print("%s - %s" % (fn.__qualname__, str(datetime.datetime.now() - now)))
        return result

    return wrap_callable

## Demonstrate

In [58]:
@time_callable
def slow_add(a: int, b: int) -> int:
    time.sleep(1)
    return a + b


slow_add(a=1, b=2)

slow_add - 0:00:01.001634


3

## Test

In [59]:
# Placeholder

# Define write_readme

## Define

In [60]:
def write_readme(outputs_directory: pathlib.Path) -> None:
    readme_path: pathlib.Path = outputs_directory.joinpath("README.md")
    paths: list[pathlib.Path] = sorted(outputs_directory.glob(pattern="*.png"))
    names: list[str] = list(map(lambda x: x.name, paths))
    stems: list[str] = list(map(lambda x: x.stem, paths))
    data: str = tz.pipe(zip(stems, names), tz.curried.map(lambda x: "# %s\n![](%s)" % x), "\n".join)
    readme_path.write_text(data=data)

## Demonstrate

In [61]:
series = pd.Series(data=np.random.randn(9))

with tempfile.TemporaryDirectory() as temporary_directory:
    outputs_directory = pathlib.Path(temporary_directory)
    series.plot()
    plt.savefig(fname=outputs_directory / "series")
    plt.close()
    write_readme(outputs_directory=outputs_directory)
    print(outputs_directory.joinpath("README.md").read_text())

# series
![](series.png)


## Test

In [62]:
# Placeholder

# Write module

## Write

In [63]:
module_path: str = "src/tools/%s.py" % module

In [64]:
%%file $module_path

import contextlib
import datetime
import logging
import pathlib
import shutil
import typing
import warnings

import numpy as np
import pandas as pd
import toolz as tz
import yaml
from matplotlib import pyplot as plt

class DocstringParser:
    def __init__(self, x: typing.Any) -> None:
        self.docstring: str = x.__doc__ or ""

    def fit(self) -> "DocstringParser":
        self.docstring_lines: pd.Series = self._get_lines()
        self.docstring_sections: pd.DataFrame = self._get_sections()
        return self

    def print_section(self, section: str = "Overview") -> None:
        print(self._get_section(section=section), end="\n" * 2)

    def print_sections(self, sections: list[str] | tuple[str] = ("Overview",)) -> None:
        for section in sections:
            self.print_section(section=section)

    def _prefix(self) -> str:
        return "Overview\n--------\n%s" % self.docstring

    def _get_lines(self) -> pd.Series:
        return pd.Series(data=self._prefix().splitlines(), name="lines").str.strip()

    def _get_sections(self) -> pd.DataFrame:
        assign_args: dict[str, typing.Callable] = {
            "has_underscores": lambda x: x["lines"].str.contains(pat=r"^[-=]{3,}$"),
            "is_header": lambda x: x["has_underscores"].shift(periods=-1).ffill(),
            "headers": lambda x: x["lines"].where(cond=x["is_header"]).ffill().fillna(value="Overview"),
        }
        agg_args: dict[str, typing.Callable] = {
            "starts": lambda x: x.index.min(),
            "stops": lambda x: x.index.max().__add__(1),
        }
        return (
            self.docstring_lines.to_frame()
            .assign(**assign_args)
            .groupby(by="headers")["lines"]
            .agg(**agg_args)
            .sort_values(by="starts")
        )

    def _get_section(self, section: str) -> str:
        if section not in self.docstring_sections.index:
            raise ValueError("Unknown section: %s" % section)
        starts_and_stops: pd.Series = self.docstring_sections.loc[section, :]
        return self.docstring_lines.iloc[slice(*starts_and_stops)].str.cat(sep="\n")

def describe_structure(x: typing.Any, indent: int = 0, max_indent: int = 2) -> None:
    prefix: str = "  " * indent
    if indent > max_indent:
        print("%s(Depth limit reached)" % prefix)
        return
    if isinstance(x, dict):
        print("%s%s with %d keys" % (prefix, type(x), len(x)))
        for key, value in x.items():  # type: tuple[typing.Any, typing.Any]
            print("%s- key: %s" % (prefix, key))
            describe_structure(x=value, indent=indent + 1, max_indent=max_indent)
    elif isinstance(x, (list, set, tuple)):
        print("%s%s with %d elements" % (prefix, type(x), len(x)))
        for i, element in enumerate(iterable=x):  # type: tuple[int, typing.Any]
            print("%s- element: %d" % (prefix, i))
            describe_structure(x=element, indent=indent + 1, max_indent=max_indent)
    else:
        print("%s%s" % (prefix, get_type_and_shape(x=x)))

def configure_logging(path: str | None, params: dict[str, typing.Any]) -> pathlib.Path | None:
    if path is not None:
        outputs_directory = pathlib.Path(path)
        shutil.rmtree(path=outputs_directory, ignore_errors=True)
        outputs_directory.mkdir()
        logging.basicConfig(filename=outputs_directory / str(datetime.date.today()), **params["basic_config_args"])
        return outputs_directory
    else:
        logging.basicConfig(**params["basic_config_args"])
        
def display_data(
    data: pd.DataFrame | pd.Series, is_in_notebook: bool, name: str | None = None, info_args: dict | None = None
) -> None:
    if name is not None:
        print("=" * int(8e1), name, "-" * int(8e1), sep="\n")
    data.info(**(info_args or {}))
    if is_in_notebook:
        print("-" * int(8e1))
        display(data)

def filter_dir(x: typing.Any, include_underscores: bool = False, include_modules: bool = False) -> pd.DataFrame:
    assign_args: dict[str, typing.Callable] = {
        "has_underscore": lambda y: y["object"].str.startswith(pat="_"),
        "type": lambda y: y["object"].apply(func=lambda z: type(getattr(x, z)).__name__),
        "is_module": lambda y: y["type"].eq(other="module"),
    }
    return (
        pd.Series(data=dir(x))
        .to_frame(name="object")
        .assign(**assign_args)
        .query(expr="has_underscore.eq(other=%s)" % include_underscores)
        .query(expr="is_module.eq(other=%s)" % include_modules)
        .set_index(keys="object")
    )

def get_shape(x: typing.Any) -> typing.Any:
    if hasattr(x, "shape"):
        return x.shape
    elif hasattr(x, "size"):
        return x.size
    elif hasattr(x, "__len__"):
        return len(x)
    else:
        return np.nan

def get_type_and_shape(x: typing.Any) -> tuple[type, typing.Any]:
    return type(x), get_shape(x=x)

@contextlib.contextmanager
def ignore_exceptions() -> typing.Iterator[None]:
    try:
        yield
    except Exception as exception:
        print(type(exception), exception)

@contextlib.contextmanager
def ignore_warnings() -> typing.Iterator[None]:
    with warnings.catch_warnings():
        warnings.filterwarnings(action="ignore")
        yield

def print_sequence(x: typing.Any, header: str = "Sequence") -> None:
    length: int = len(x)
    length_of_length: int = len(str(length))
    sequence: str = tz.pipe(
        enumerate(iterable=x), tz.curried.map(lambda y: f"{y[0]:0{length_of_length}d}. {y[1]}"), "\n".join
    )
    print("%s (%d):\n%s" % (header, length, sequence))

def print_shapes(x: typing.Any, include_types: bool = False, **kwargs) -> None:
    print(*map(get_type_and_shape if include_types else get_shape, x), **kwargs)

def print_type_and_return(x: typing.Any) -> typing.Any:
    print(type(x))
    return x

def read_in_params(path: str) -> dict[str, typing.Any]:
    with open(file=path) as file:
        params: dict[str, typing.Any] = yaml.safe_load(stream=file)
    return params

def save_show_and_close(outputs_directory: pathlib.Path | None, filename: str | None, is_in_notebook: bool) -> None:
    if outputs_directory is not None and filename is not None:
        plt.savefig(fname=outputs_directory / filename, bbox_inches="tight")
    if is_in_notebook:
        plt.show()
    plt.close()

def time_callable(fn: typing.Callable) -> typing.Callable:
    def wrap_callable(*args, **kwargs):
        now = datetime.datetime.now()
        result = fn(*args, **kwargs)
        print("%s - %s" % (fn.__qualname__, str(datetime.datetime.now() - now)))
        return result

    return wrap_callable

def write_readme(outputs_directory: pathlib.Path) -> None:
    readme_path: pathlib.Path = outputs_directory.joinpath("README.md")
    paths: list[pathlib.Path] = sorted(outputs_directory.glob(pattern="*.png"))
    names: list[str] = list(map(lambda x: x.name, paths))
    stems: list[str] = list(map(lambda x: x.stem, paths))
    data: str = tz.pipe(zip(stems, names), tz.curried.map(lambda x: "# %s\n![](%s)" % x), "\n".join)
    readme_path.write_text(data=data)

Overwriting src/tools/utils.py


## Format

In [65]:
!ruff format $module_path

1 file reformatted


## Check with ruff

In [66]:
!ruff check $module_path

F821 Undefined name `display`
   --> src\tools\utils.py:101:9
    |
 99 |     if is_in_notebook:
100 |         print("-" * int(8e1))
101 |         display(data)
    |         ^^^^^^^
    |

Found 1 error.


## Check with ty

In [67]:
!ty check $module_path

error[unresolved-reference]: Name `display` used when not defined
   --> src\tools\utils.py:101:9
    |
101 |         display(data)
    |         ^^^^^^^
    |

error[unresolved-attribute]: Object of type `(...) -> Unknown` has no attribute `__qualname__`
   --> src\tools\utils.py:186:28
    |
186 |         print("%s - %s" % (fn.__qualname__, str(datetime.datetime.now() - now)))
    |                            ^^^^^^^^^^^^^^^
    |
help: Function objects have a `__qualname__` attribute, but not all callable objects are functions
help: See this FAQ for more information: <https://docs.astral.sh/ty/reference/typing-faq/#why-does-ty-say-callable-has-no-attribute-__name__>

Found 2 diagnostics
